[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/053.%20The%20EOQ%20with%20Planned%20Shortages%20Problem/P53-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 53. The EOQ with Planned Shortages Problem

## Tier 1: The Pen & Paper Method (Mixed-Integer Programming Formulation)

### Key Assumptions
- Demand is constant and known (36,000 units/year)
- Ordering cost is fixed per order ($150)
- Holding cost is linear with inventory level ($8/unit/year)
- Shortage cost is linear with shortage level ($12/unit/year)
- Planned shortages are allowed and backordered
- Lead time is zero (instantaneous replenishment)

### Approach (Step-by-Step)
1. **Mathematical Model Formulation**: Define decision variables and constraints
2. **Objective Function**: Minimize total cost (ordering + holding + shortage)
3. **Optimal Solution**: Derive analytical solutions using calculus
4. **Numerical Example**: Apply to Meridian Electronics case
5. **Visualization**: Plot inventory levels over time

### What to Look for in the Results
- Optimal order quantity (Q*) that balances all costs
- Optimal shortage level (S*) that minimizes total cost
- Maximum inventory level (Imax*) before shortages begin
- Time periods with positive inventory vs shortages
- Total cost breakdown across all cost components

### Concrete Example (from the source)
**Meridian Electronics Distribution Center**
- Annual demand (D): 36,000 units
- Ordering cost (K): $150 per order
- Holding cost (h): $8 per unit per year
- Shortage cost (p): $12 per unit per year

In [1]:
# Import required libraries for mathematical optimization and visualization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize_scalar
import pandas as pd

# Set professional plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully for EOQ with Planned Shortages analysis")

In [2]:
# Define the EOQ with Planned Shortages problem class
class EOQWithShortages:
    """
    EOQ with Planned Shortages Problem Solver
    Implements the mathematical formulation for optimal inventory management
    with controlled stockouts and backordering.
    """
    
    def __init__(self, demand, ordering_cost, holding_cost, shortage_cost):
        """
        Initialize EOQ with Shortages problem parameters
        
        Args:
            demand: Annual demand rate (units/year)
            ordering_cost: Fixed cost per order ($)
            holding_cost: Annual holding cost per unit ($/unit/year)
            shortage_cost: Annual shortage cost per unit ($/unit/year)
        """
        self.D = demand  # Annual demand
        self.K = ordering_cost  # Ordering cost per order
        self.h = holding_cost  # Holding cost per unit per year
        self.p = shortage_cost  # Shortage cost per unit per year
        
    def calculate_optimal_solution(self):
        """
        Calculate optimal EOQ with shortages using analytical formulas
        
        Mathematical formulation:
        Q* = sqrt(2*D*K*(h+p)/(h*p))
        S* = Q* * h/(h+p)
        Imax* = Q* * p/(h+p)
        """
        # Calculate optimal order quantity
        Q_optimal = np.sqrt(2 * self.D * self.K * (self.h + self.p) / (self.h * self.p))
        
        # Calculate optimal shortage level
        S_optimal = Q_optimal * self.h / (self.h + self.p)
        
        # Calculate maximum inventory level
        Imax_optimal = Q_optimal * self.p / (self.h + self.p)
        
        # Calculate cycle time
        cycle_time = Q_optimal / self.D
        
        # Calculate time with positive inventory
        t_positive = Imax_optimal / self.D
        
        # Calculate time with shortages
        t_shortage = S_optimal / self.D
        
        # Calculate total annual cost
        ordering_cost = self.D * self.K / Q_optimal
        holding_cost = Imax_optimal**2 * self.h / (2 * self.D)
        shortage_cost = S_optimal**2 * self.p / (2 * self.D)
        total_cost = ordering_cost + holding_cost + shortage_cost
        
        return {
            'Q_optimal': Q_optimal,
            'S_optimal': S_optimal,
            'Imax_optimal': Imax_optimal,
            'cycle_time': cycle_time,
            't_positive': t_positive,
            't_shortage': t_shortage,
            'ordering_cost': ordering_cost,
            'holding_cost': holding_cost,
            'shortage_cost': shortage_cost,
            'total_cost': total_cost
        }
    
    def calculate_total_cost(self, Q, S):
        """
        Calculate total cost for given order quantity and shortage level
        
        Total Cost = (D*K)/Q + ((Q-S)^2 * h)/(2*D) + (S^2 * p)/(2*D)
        """
        if Q <= 0 or S < 0 or S > Q:
            return float('inf')
            
        ordering_cost = self.D * self.K / Q
        holding_cost = (Q - S)**2 * self.h / (2 * self.D)
        shortage_cost = S**2 * self.p / (2 * self.D)
        
        return ordering_cost + holding_cost + shortage_cost

print("EOQ with Shortages class defined successfully")

In [ ]:
# Initialize the problem with Meridian Electronics parameters
meridian_eoq = EOQWithShortages(
    demand=36000,        # 36,000 units/year
    ordering_cost=150,   # $150 per order
    holding_cost=8,       # $8 per unit per year
    shortage_cost=12      # $12 per unit per year
)

# Calculate optimal solution
optimal_solution = meridian_eoq.calculate_optimal_solution()

# Display results in a formatted way
print("=" * 60)
print("MERIDIAN ELECTRONICS - EOQ WITH PLANNED SHORTAGES")
print("=" * 60)
print(f"Problem Parameters:")
print(f"  Annual Demand (D): {meridian_eoq.D:,} units/year")
print(f"  Ordering Cost (K): ${meridian_eoq.K:,.2f} per order")
print(f"  Holding Cost (h): ${meridian_eoq.h:,.2f} per unit/year")
print(f"  Shortage Cost (p): ${meridian_eoq.p:,.2f} per unit/year")
print()
print(f"Optimal Solution:")
print(f"  Order Quantity (Q*): {optimal_solution['Q_optimal']:,.0f} units")
print(f"  Shortage Level (S*): {optimal_solution['S_optimal']:,.0f} units")
print(f"  Max Inventory (Imax*): {optimal_solution['Imax_optimal']:,.0f} units")
print(f"  Cycle Time: {optimal_solution['cycle_time']*365:.1f} days")
print(f"  Time with Inventory: {optimal_solution['t_positive']*365:.1f} days")
print(f"  Time with Shortages: {optimal_solution['t_shortage']*365:.1f} days")
print()
print(f"Cost Analysis:")
print(f"  Ordering Cost: ${optimal_solution['ordering_cost']:,.2f}")
print(f"  Holding Cost: ${optimal_solution['holding_cost']:,.2f}")
print(f"  Shortage Cost: ${optimal_solution['shortage_cost']:,.2f}")
print(f"  Total Annual Cost: ${optimal_solution['total_cost']:,.2f}")
print("=" * 60)

In [ ]:
# Create inventory level visualization over one cycle
def plot_inventory_cycle(eoq_problem, solution):
    """
    Visualize inventory levels over one complete cycle
    Shows the pattern of inventory depletion and shortage accumulation
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    # Time points for smooth curves
    n_points = 100
    
    # Time with positive inventory (0 to t_positive)
    t1 = np.linspace(0, solution['t_positive'], n_points//2)
    # Time with shortages (t_positive to cycle_time)
    t2 = np.linspace(solution['t_positive'], solution['cycle_time'], n_points//2)
    
    # Inventory levels during positive inventory period
    inventory_positive = solution['Imax_optimal'] * (1 - t1 / solution['t_positive'])
    
    # Inventory levels during shortage period (negative values)
    inventory_shortage = -solution['S_optimal'] * ((t2 - solution['t_positive']) / solution['t_shortage'])
    
    # Combine time and inventory arrays
    time_combined = np.concatenate([t1, t2])
    inventory_combined = np.concatenate([inventory_positive, inventory_shortage])
    
    # Plot 1: Inventory Level Over Time
    ax1.fill_between(time_combined * 365, 0, inventory_combined, 
                     where=(inventory_combined >= 0), alpha=0.3, color='green', label='Positive Inventory')
    ax1.fill_between(time_combined * 365, 0, inventory_combined, 
                     where=(inventory_combined < 0), alpha=0.3, color='red', label='Shortages')
    ax1.plot(time_combined * 365, inventory_combined, 'b-', linewidth=2, label='Inventory Level')
    ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax1.axhline(y=solution['Imax_optimal'], color='green', linestyle='--', alpha=0.5, label=f'Max Inventory: {solution["Imax_optimal"]:.0f}')
    ax1.axhline(y=-solution['S_optimal'], color='red', linestyle='--', alpha=0.5, label=f'Max Shortage: {solution["S_optimal"]:.0f}')
    
    ax1.set_xlabel('Time (days)')
    ax1.set_ylabel('Inventory Level (units)')
    ax1.set_title('EOQ with Planned Shortages - Inventory Cycle Pattern')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Cost Components Breakdown
    cost_components = ['Ordering Cost', 'Holding Cost', 'Shortage Cost']
    cost_values = [solution['ordering_cost'], solution['holding_cost'], solution['shortage_cost']]
    colors = ['blue', 'green', 'red']
    
    bars = ax2.bar(cost_components, cost_values, color=colors, alpha=0.7)
    ax2.set_ylabel('Annual Cost ($)')
    ax2.set_title('Cost Component Breakdown')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, value in zip(bars, cost_values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'${value:,.0f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Generate the visualization
plot_inventory_cycle(meridian_eoq, optimal_solution)

In [ ]:
# Sensitivity analysis - How optimal solution changes with parameter variations
def sensitivity_analysis(eoq_problem):
    """
    Perform sensitivity analysis on key parameters
    Shows how the optimal solution changes with parameter variations
    """
    # Parameter ranges for sensitivity analysis (±20% variation)
    demand_range = np.linspace(eoq_problem.D * 0.8, eoq_problem.D * 1.2, 5)
    holding_cost_range = np.linspace(eoq_problem.h * 0.8, eoq_problem.h * 1.2, 5)
    shortage_cost_range = np.linspace(eoq_problem.p * 0.8, eoq_problem.p * 1.2, 5)
    
    # Store results
    sensitivity_results = {
        'demand_variation': [],
        'holding_cost_variation': [],
        'shortage_cost_variation': []
    }
    
    # Demand sensitivity
    for demand in demand_range:
        temp_eoq = EOQWithShortages(demand, eoq_problem.K, eoq_problem.h, eoq_problem.p)
        solution = temp_eoq.calculate_optimal_solution()
        sensitivity_results['demand_variation'].append({
            'parameter': demand,
            'Q_optimal': solution['Q_optimal'],
            'S_optimal': solution['S_optimal'],
            'total_cost': solution['total_cost']
        })
    
    # Holding cost sensitivity
    for h_cost in holding_cost_range:
        temp_eoq = EOQWithShortages(eoq_problem.D, eoq_problem.K, h_cost, eoq_problem.p)
        solution = temp_eoq.calculate_optimal_solution()
        sensitivity_results['holding_cost_variation'].append({
            'parameter': h_cost,
            'Q_optimal': solution['Q_optimal'],
            'S_optimal': solution['S_optimal'],
            'total_cost': solution['total_cost']
        })
    
    # Shortage cost sensitivity
    for s_cost in shortage_cost_range:
        temp_eoq = EOQWithShortages(eoq_problem.D, eoq_problem.K, eoq_problem.h, s_cost)
        solution = temp_eoq.calculate_optimal_solution()
        sensitivity_results['shortage_cost_variation'].append({
            'parameter': s_cost,
            'Q_optimal': solution['Q_optimal'],
            'S_optimal': solution['S_optimal'],
            'total_cost': solution['total_cost']
        })
    
    return sensitivity_results, demand_range, holding_cost_range, shortage_cost_range

# Perform sensitivity analysis
sensitivity_results, demand_range, holding_cost_range, shortage_cost_range = sensitivity_analysis(meridian_eoq)

# Create sensitivity analysis plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('EOQ with Planned Shortages - Sensitivity Analysis', fontsize=16, fontweight='bold')

# Row 1: Order Quantity sensitivity
axes[0, 0].plot(demand_range, [r['Q_optimal'] for r in sensitivity_results['demand_variation']], 'bo-')
axes[0, 0].set_xlabel('Annual Demand (units)')
axes[0, 0].set_ylabel('Optimal Order Quantity (units)')
axes[0, 0].set_title('Q* vs Demand')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(holding_cost_range, [r['Q_optimal'] for r in sensitivity_results['holding_cost_variation']], 'go-')
axes[0, 1].set_xlabel('Holding Cost ($/unit/year)')
axes[0, 1].set_ylabel('Optimal Order Quantity (units)')
axes[0, 1].set_title('Q* vs Holding Cost')
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(shortage_cost_range, [r['Q_optimal'] for r in sensitivity_results['shortage_cost_variation']], 'ro-')
axes[0, 2].set_xlabel('Shortage Cost ($/unit/year)')
axes[0, 2].set_ylabel('Optimal Order Quantity (units)')
axes[0, 2].set_title('Q* vs Shortage Cost')
axes[0, 2].grid(True, alpha=0.3)

# Row 2: Shortage Level sensitivity
axes[1, 0].plot(demand_range, [r['S_optimal'] for r in sensitivity_results['demand_variation']], 'bo-')
axes[1, 0].set_xlabel('Annual Demand (units)')
axes[1, 0].set_ylabel('Optimal Shortage Level (units)')
axes[1, 0].set_title('S* vs Demand')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(holding_cost_range, [r['S_optimal'] for r in sensitivity_results['holding_cost_variation']], 'go-')
axes[1, 1].set_xlabel('Holding Cost ($/unit/year)')
axes[1, 1].set_ylabel('Optimal Shortage Level (units)')
axes[1, 1].set_title('S* vs Holding Cost')
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(shortage_cost_range, [r['S_optimal'] for r in sensitivity_results['shortage_cost_variation']], 'ro-')
axes[1, 2].set_xlabel('Shortage Cost ($/unit/year)')
axes[1, 2].set_ylabel('Optimal Shortage Level (units)')
axes[1, 2].set_title('S* vs Shortage Cost')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare with traditional EOQ (no shortages allowed)
def compare_with_traditional_eoq(eoq_problem):
    """
    Compare EOQ with planned shortages vs traditional EOQ (no shortages)
    Demonstrates the cost benefits of allowing planned shortages
    """
    # Traditional EOQ formula
    Q_traditional = np.sqrt(2 * eoq_problem.D * eoq_problem.K / eoq_problem.h)
    
    # Traditional EOQ costs
    traditional_ordering_cost = eoq_problem.D * eoq_problem.K / Q_traditional
    traditional_holding_cost = Q_traditional * eoq_problem.h / 2
    traditional_total_cost = traditional_ordering_cost + traditional_holding_cost
    
    # EOQ with shortages costs (from previous calculation)
    shortage_solution = eoq_problem.calculate_optimal_solution()
    
    # Create comparison table
    comparison_data = {
        'Metric': [
            'Order Quantity (units)',
            'Maximum Inventory (units)',
            'Ordering Cost ($/year)',
            'Holding Cost ($/year)',
            'Shortage Cost ($/year)',
            'Total Cost ($/year)'
        ],
        'Traditional EOQ': [
            f"{Q_traditional:.0f}",
            f"{Q_traditional:.0f}",
            f"{traditional_ordering_cost:.2f}",
            f"{traditional_holding_cost:.2f}",
            "0.00",
            f"{traditional_total_cost:.2f}"
        ],
        'EOQ with Shortages': [
            f"{shortage_solution['Q_optimal']:.0f}",
            f"{shortage_solution['Imax_optimal']:.0f}",
            f"{shortage_solution['ordering_cost']:.2f}",
            f"{shortage_solution['holding_cost']:.2f}",
            f"{shortage_solution['shortage_cost']:.2f}",
            f"{shortage_solution['total_cost']:.2f}"
        ]
    }
    
    comparison_df = pd.DataFrame(comparison_data)
    
    print("=" * 80)
    print("COMPARISON: TRADITIONAL EOQ vs EOQ WITH PLANNED SHORTAGES")
    print("=" * 80)
    print(comparison_df.to_string(index=False))
    print()
    
    cost_savings = traditional_total_cost - shortage_solution['total_cost']
    percentage_savings = (cost_savings / traditional_total_cost) * 100
    
    print(f"Cost Savings with Planned Shortages: ${cost_savings:.2f} ({percentage_savings:.2f}%)")
    print(f"Service Level: {(1 - shortage_solution['S_optimal']/shortage_solution['Q_optimal'])*100:.1f}%")
    print("=" * 80)
    
    return comparison_df, cost_savings, percentage_savings

# Perform comparison
comparison_df, cost_savings, percentage_savings = compare_with_traditional_eoq(meridian_eoq)

In [ ]:
# What-if analysis: Different shortage cost scenarios
def what_if_shortage_cost_analysis():
    """
    Analyze how different shortage cost levels affect the optimal solution
    Demonstrates the trade-off between holding costs and shortage costs
    """
    # Different shortage cost scenarios
    shortage_scenarios = [
        (6, "Low Shortage Cost"),      # p = $6 (equal to holding cost)
        (12, "Base Case"),             # p = $12 (base case)
        (24, "High Shortage Cost"),    # p = $24 (double holding cost)
        (48, "Very High Shortage Cost") # p = $48 (4x holding cost)
    ]
    
    results = []
    
    for shortage_cost, scenario_name in shortage_scenarios:
        eoq_scenario = EOQWithShortages(36000, 150, 8, shortage_cost)
        solution = eoq_scenario.calculate_optimal_solution()
        
        results.append({
            'Scenario': scenario_name,
            'Shortage Cost': shortage_cost,
            'Q*': solution['Q_optimal'],
            'S*': solution['S_optimal'],
            'Imax*': solution['Imax_optimal'],
            'Total Cost': solution['total_cost'],
            'Service Level %': (1 - solution['S_optimal']/solution['Q_optimal']) * 100
        })
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)
    
    print("=" * 100)
    print("WHAT-IF ANALYSIS: IMPACT OF SHORTAGE COST ON OPTIMAL SOLUTION")
    print("=" * 100)
    print(results_df.round(2).to_string(index=False))
    print("=" * 100)
    
    # Visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('What-If Analysis: Shortage Cost Impact', fontsize=16, fontweight='bold')
    
    # Order Quantity vs Shortage Cost
    ax1.plot(results_df['Shortage Cost'], results_df['Q*'], 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Shortage Cost ($/unit/year)')
    ax1.set_ylabel('Optimal Order Quantity (units)')
    ax1.set_title('Order Quantity vs Shortage Cost')
    ax1.grid(True, alpha=0.3)
    
    # Shortage Level vs Shortage Cost
    ax2.plot(results_df['Shortage Cost'], results_df['S*'], 'ro-', linewidth=2, markersize=8)
    ax2.set_xlabel('Shortage Cost ($/unit/year)')
    ax2.set_ylabel('Optimal Shortage Level (units)')
    ax2.set_title('Shortage Level vs Shortage Cost')
    ax2.grid(True, alpha=0.3)
    
    # Total Cost vs Shortage Cost
    ax3.plot(results_df['Shortage Cost'], results_df['Total Cost'], 'go-', linewidth=2, markersize=8)
    ax3.set_xlabel('Shortage Cost ($/unit/year)')
    ax3.set_ylabel('Total Annual Cost ($)')
    ax3.set_title('Total Cost vs Shortage Cost')
    ax3.grid(True, alpha=0.3)
    
    # Service Level vs Shortage Cost
    ax4.plot(results_df['Shortage Cost'], results_df['Service Level %'], 'mo-', linewidth=2, markersize=8)
    ax4.set_xlabel('Shortage Cost ($/unit/year)')
    ax4.set_ylabel('Service Level (%)')
    ax4.set_title('Service Level vs Shortage Cost')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return results_df

# Run what-if analysis
what_if_results = what_if_shortage_cost_analysis()

### Why This Tier Exists vs Earlier Approaches

**Mathematical Foundation**: This tier provides the theoretical foundation for EOQ with planned shortages, offering:
- **Exact optimal solutions** through analytical derivation
- **Mathematical proof** of optimality conditions
- **Benchmark** for evaluating heuristic and metaheuristic approaches
- **Sensitivity analysis** capabilities for parameter variations

### Pros vs Cons

**Advantages:**
- ✅ **Guaranteed optimality** - Provides provably optimal solution
- ✅ **Instant computation** - Analytical formulas give immediate results
- ✅ **Complete insight** - Shows relationships between all parameters
- ✅ **Benchmark quality** - Standard against which other methods are measured
- ✅ **No approximation error** - Exact mathematical solution

**Disadvantages:**
- ❌ **Limited assumptions** - Requires constant demand, linear costs
- ❌ **No discrete constraints** - Cannot handle integer requirements directly
- ❌ **Single product focus** - Doesn't extend easily to multi-product cases
- ❌ **No stochastic elements** - Cannot handle demand uncertainty
- ❌ **Simplified cost structure** - Real-world complexities often violate assumptions

### When to Use This Tier

**Use Tier 1 when:**
- Problem parameters are stable and well-known
- Demand is relatively constant and predictable
- Cost structures are linear and well-defined
- You need guaranteed optimal solutions
- Serving as a benchmark for other methods
- Educational purposes to understand the theoretical foundation

**Consider other tiers when:**
- Demand is stochastic or seasonal
- Multiple products with shared resources
- Complex constraints (capacity, discounts, etc.)
- Need for real-time decision making
- Large-scale optimization problems

### Key Insights from Mathematical Formulation

1. **Cost Trade-off**: The optimal solution balances ordering, holding, and shortage costs
2. **Service Level Impact**: Higher shortage costs lead to lower planned shortages and higher service levels
3. **Economic Order Quantity**: The optimal order quantity increases when shortage costs are considered
4. **Cycle Structure**: Each cycle consists of positive inventory period followed by shortage period
5. **Cost Savings**: Allowing planned shortages can reduce total costs compared to traditional EOQ

In [ ]:
# Final summary and key takeaways
print("=" * 80)
print("EOQ WITH PLANNED SHORTAGES - TIER 1 SUMMARY")
print("=" * 80)
print()
print("📊 MATHEMATICAL FORMULATION RESULTS:")
print(f"   • Optimal Order Quantity: {optimal_solution['Q_optimal']:,.0f} units")
print(f"   • Optimal Shortage Level: {optimal_solution['S_optimal']:,.0f} units")
print(f"   • Maximum Inventory: {optimal_solution['Imax_optimal']:,.0f} units")
print(f"   • Cycle Time: {optimal_solution['cycle_time']*365:.1f} days")
print(f"   • Service Level: {(1-optimal_solution['S_optimal']/optimal_solution['Q_optimal'])*100:.1f}%")
print()
print(f"💰 COST ANALYSIS:")
print(f"   • Ordering Cost: ${optimal_solution['ordering_cost']:,.2f} ({optimal_solution['ordering_cost']/optimal_solution['total_cost']*100:.1f}%)")
print(f"   • Holding Cost: ${optimal_solution['holding_cost']:,.2f} ({optimal_solution['holding_cost']/optimal_solution['total_cost']*100:.1f}%)")
print(f"   • Shortage Cost: ${optimal_solution['shortage_cost']:,.2f} ({optimal_solution['shortage_cost']/optimal_solution['total_cost']*100:.1f}%)")
print(f"   • Total Annual Cost: ${optimal_solution['total_cost']:,.2f}")
print()
print(f"💡 KEY INSIGHTS:")
print(f"   • Cost Savings vs Traditional EOQ: ${cost_savings:.2f} ({percentage_savings:.2f}%)")
print(f"   • Planned shortages reduce total costs while maintaining service")
print(f"   • Mathematical formulation provides exact optimal solution")
print(f"   • Sensitivity analysis shows robust parameter relationships")
print()
print("=" * 80)
print("✅ TIER 1 COMPLETE - Mathematical foundation established for EOQ with Planned Shortages")
print("=" * 80)